## test sample ratio needed for training.

In [ ]:
import numpy as np
x=np.array([])
total_num = 57
train = np.array([])
valid = np.array([])
# train_num = int(0.8 * total_num)
train_num = 57

# choice = 'macro5'
choice = 'micro100'

for i in range(train_num):
    if choice=='macro5':
        path = f"../data/Fip35/unzip_all/regen/{choice}/macro5_{i}"
    else:
        path = f"../data/Fip35/unzip_all/regen/{choice}/traj_{i}"
    # path = f"data/Fip35/macro5/macro5_{i}"
    single_file = np.loadtxt(path,dtype=int)
    if single_file.shape[0] % 5 != 0:
        single_file = single_file[:-(single_file.shape[0] % 5)]
    print(single_file.shape)
    # if i == 23:
    #     single_file = single_file[:32585] # continue # 23 has entry, 32586, which cannot be subsampled by 5
    train = np.append(train, single_file[:int(len(single_file)/3)])

In [ ]:
from msmbuilder.msm import MarkovStateModel
for lagtime in range(50, 1001, 50):
    msm = MarkovStateModel(lag_time=lagtime, n_timescales=10, verbose=False, reversible_type="transpose")
    msm.fit(list(train))
    print(lagtime, end=" ")
    for i in range(4):
        print(msm.timescales_[i], end=" ")
    print("\n", end="")

# Fip35 dataset

In [11]:
import numpy as np
from pyemma import msm, plots
import matplotlib.pyplot as plt
import sys
import itertools
sys.path.append('/home/zengwenqi/projects/MD_code2')
from msmbuilder.msm import MarkovStateModel
import numpy
import pandas as pd


## residence probability

In [13]:
task = 'Fip35_macro5'
# task = 'Fip35_micro100' 
# task = 'MacroAssignment'
datapath = f'/home/zengwenqi/projects/MD_code2/data/{task}/'
train=np.loadtxt(datapath + 'train', dtype=int).reshape(-1)
test = np.loadtxt(datapath + 'test', dtype=int).reshape(-1)


In [14]:
interval = 5
epoch = 40
task = 'Fip35_micro100' 
# task = 'Fip35_macro5' 
# task = 'Fip35_macro'
# task = 'Fip35_micro' 
pred_path = f'/home/zengwenqi/projects/MD_code2/results/{task}/trans_gpt/Label0.0_window50_interval{interval}_lr0.0005_emb_dim128_l100_block2_scheduled/category/epoch{epoch}_100000_valid_interval{interval}_gen20'
# pred_path = "/home/zengwenqi/projects/MD_code2/results/Fip35_macro/trans_gpt/Label0.0_window50_interval1_lr0.0005_emb_dim128_l100_block2_scheduled_false20/category/epoch40_100000_valid_interval1"
# pred_path_lstm = f'/home/zengwenqi/projects/MD_code2/results/Fip35_micro100/lstm/Label0.0_sparse50_interval{interval}_lr0.001_emb_dim128_l100_units512_emb128_no_pos/category/epoch{epoch}_100000_valid_interval1_gen20'
def load_data(path):
    data = np.loadtxt(path, dtype= int)
    # data = Rm_peaks_steps(data, 100)
    return data
pred_traj = [load_data(pred_path + f'/no_gen_pos_prediction_{i}') for i in range(20)]

In [17]:
train.shape

(4432585,)

In [15]:
import numpy as np

def calculate_residence_probability_from_data(data, num_states, max_residence_time):
    residence_counts = np.zeros((num_states, max_residence_time))
    current_state = data[0]
    current_count = 0
    
    for i, state in enumerate(data):
        if state == current_state:
            current_count += 1
        else:
            residence_time = min(current_count, max_residence_time)
            residence_counts[current_state, residence_time - 1] += 1
            current_state = state
            current_count = 1

    residence_time = min(current_count, max_residence_time)
    residence_counts[current_state, residence_time - 1] += 1
    residence_probabilities = residence_counts / residence_counts.sum(axis=1, keepdims=True)
    
    return residence_probabilities




In [ ]:
num_states = 5  # Number of distinct states
max_residence_time = 5  # Maximum residence time to consider

residence_probs = calculate_residence_probability_from_data(train, num_states, max_residence_time)

print("Residence probabilities:")
print(residence_probs)

## ITS

In [12]:
# for lstm experiment
# task = 'Fip35_macro5'
task = 'Fip35_micro100' 
# task = 'MacroAssignment'
datapath = f'/home/zengwenqi/projects/MD_code2/data/{task}/'
train=np.loadtxt(datapath + 'train', dtype=int).reshape(-1)
test = np.loadtxt(datapath + 'test', dtype=int).reshape(-1)



In [8]:
for lagtime in range(50, 1001, 50):
    msm = MarkovStateModel(lag_time=lagtime, n_timescales=10, verbose=False, reversible_type="transpose")
    msm.fit(list(train[0:int(len(train)/2)]))
    print(lagtime, end=" ")
    for i in range(4):
        print(msm.timescales_[i], end=" ")
    print("\n", end="")


50 25598.590516799308 4882.953143927023 472.7807741738011 251.02507827419785 
100 39770.54419474935 6437.969098873943 650.6167825797978 376.15507635379197 
150 49762.67611316283 7781.249351947129 767.0525086645075 467.2035784514878 
200 59531.66704415224 8492.279847525422 840.7218064539456 561.0164574753497 
250 68460.18066123601 8940.882093166712 897.279476607843 620.7799776255908 
300 75090.94909386645 9290.151891543828 934.5682443257584 673.8207756976827 
350 79885.5812170526 9583.625473631024 964.3966264075643 727.4844397240175 
400 84759.74446829299 9834.156923127435 990.0475315539155 779.4701614964217 
450 89085.84395598162 10081.508699592538 1017.5029674587144 830.1650911802868 
500 94116.15747883962 10293.635942308805 1039.456455718359 884.5522203689972 
550 98078.70021610323 10461.171747756593 1060.7762738346573 922.7195993996951 
600 102900.40831134225 10554.974391635165 1078.8198842616935 959.687152880391 
650 106954.48596386156 10613.520431467912 1097.8021991245635 1006.029

In [3]:


# compute microstate TPM at lagtime 50 frames
# for FIP35, 5 frames = 1 ns
for lagtime in range(50, 1001, 50):
    msm = MarkovStateModel(lag_time=lagtime, n_timescales=10, verbose=False, reversible_type="transpose")
    msm.fit(list(train))
    print(lagtime, end=" ")
    for i in range(4):
        print(msm.timescales_[i], end=" ")
    print("\n", end="")


50 21233.440317711447 4230.065988372829 466.8445430570313 314.9547202438035 
100 31720.899925358128 5580.670817163736 654.6712336960916 454.5245907507514 
150 38472.97793498208 6796.687758497262 783.8333614989123 539.1256632896103 
200 44444.75722981899 7466.865592729466 861.9894174804123 624.8215277891867 
250 49218.417631428085 7907.776635706142 922.7701792865194 685.3249542786176 
300 52433.137494019305 8274.950932687841 970.4183366129058 744.810965640534 
350 55038.9780163403 8573.703252691412 1002.2802256894363 801.1279291772425 
400 57265.3844409055 8850.16294631627 1026.5536700451025 851.4541286902346 
450 59316.914153427584 9111.13423249166 1053.4710582658736 898.7791036721467 
500 61416.116000805305 9326.98657749367 1075.6928716342884 943.6272172885368 
550 63116.87113238805 9525.515130755246 1093.1491316530276 971.1456938144169 
600 64995.442248814594 9662.460621061613 1111.2628409831368 989.7732576783822 
650 66271.72168003244 9759.02298573093 1124.227497352199 1015.60225114

In [ ]:
for lagtime in range(50, 1001, 50):
    msm = MarkovStateModel(lag_time=lagtime, n_timescales=10, verbose=False, reversible_type="transpose")
    msm.fit(list(train))
    print(lagtime, end=" ")
    for i in range(4):
        print(msm.timescales_[i], end=" ")
    print("\n", end="")


# code siqin

In [ ]:


ktrajs = []
for i in range(56):
    this_traj = numpy.loadtxt("/home/zengwenqi/projects/MD_code2/data/Fip35/micro1000/traj_"+str(i))
    ktrajs.append(this_traj)

# compute microstate TPM at lagtime 50 frames
# for FIP35, 5 frames = 1 ns
for lagtime in range(50, 1001, 50):
    msm = MarkovStateModel(lag_time=lagtime, n_timescales=10, verbose=False, reversible_type="transpose")
    msm.fit(list(ktrajs))
    print(lagtime, end=" ")
    for i in range(10):
        print(msm.timescales_[i], end=" ")
    print("\n", end="")


50 40171.84231366875 5047.420634115022 732.550208576934 452.04620262422577 345.75367035440274 316.70979900724967 195.01636205648907 185.9822013493888 139.9160555913451 125.61063591337364 
100 50894.79139142699 6494.3669727530305 913.8709355338315 642.7482458854804 471.6565232575656 442.9796605691849 271.5138077946583 255.18992693865036 234.24134623248392 197.63931508510856 
150 56866.97424682021 7835.413541917838 1029.8297981409498 770.3354217747705 550.2249927926783 515.2598744922004 337.85756899128864 298.93275207965337 294.5688098749272 270.8867011535454 
200 61122.79264976393 8539.859890726255 1097.5378530273192 876.190322003842 614.341781320407 563.480670149568 388.9440336472718 354.21308333027434 346.3840190158686 316.7598240820625 
250 63999.08378128431 8979.178416944838 1146.0435190432272 949.5604095914937 671.5746638706146 591.0764220391807 438.48975766867494 402.62069976639447 398.6864822387821 340.54927514703513 
300 66104.46657782537 9340.672440151846 1183.3790855719717 101

In [8]:
# task = 'Fip35_macro5'
task = 'Fip35_micro100' 
# task = 'MacroAssignment'
datapath = f'/home/zengwenqi/projects/MD_code2/data/{task}/'
train=np.loadtxt(datapath + 'train', dtype=int).reshape(-1)
test = np.loadtxt(datapath + 'test', dtype=int).reshape(-1)

for lagtime in range(50, 2001, 50):
    msm = MarkovStateModel(lag_time=lagtime, n_timescales=4, verbose=False, reversible_type="transpose")
    msm.fit(list(train))
    print(lagtime, end=" ")
    for i in range(4):
        print(msm.timescales_[i], end=" ")
    print("\n", end="")

50 21233.440317711447 4230.065988372829 466.8445430570313 314.9547202438035 
100 31720.899925358128 5580.670817163736 654.6712336960916 454.5245907507514 
150 38472.97793498208 6796.687758497262 783.8333614989123 539.1256632896103 
200 44444.75722981899 7466.865592729466 861.9894174804123 624.8215277891867 
250 49218.417631428085 7907.776635706142 922.7701792865194 685.3249542786176 
300 52433.137494019305 8274.950932687841 970.4183366129058 744.810965640534 
350 55038.9780163403 8573.703252691412 1002.2802256894363 801.1279291772425 
400 57265.3844409055 8850.16294631627 1026.5536700451025 851.4541286902346 
450 59316.914153427584 9111.13423249166 1053.4710582658736 898.7791036721467 
500 61416.116000805305 9326.98657749367 1075.6928716342884 943.6272172885368 
550 63116.87113238805 9525.515130755246 1093.1491316530276 971.1456938144169 
600 64995.442248814594 9662.460621061613 1111.2628409831368 989.7732576783822 
650 66271.72168003244 9759.02298573093 1124.227497352199 1015.60225114

In [14]:
interval = 5
epoch = 40
task = 'Fip35_micro100' 
# task = 'Fip35_macro5' 
# task = 'Fip35_macro'
# task = 'Fip35_micro' 
pred_path = f'/home/zengwenqi/projects/MD_code2/results/{task}/trans_gpt/Label0.0_window50_interval{interval}_lr0.0005_emb_dim128_l100_block2_scheduled/category/epoch{epoch}_100000_valid_interval{interval}_gen20'
# pred_path = "/home/zengwenqi/projects/MD_code2/results/Fip35_macro/trans_gpt/Label0.0_window50_interval1_lr0.0005_emb_dim128_l100_block2_scheduled_false20/category/epoch40_100000_valid_interval1"
# pred_path_lstm = f'/home/zengwenqi/projects/MD_code2/results/Fip35_micro100/lstm/Label0.0_sparse50_interval{interval}_lr0.001_emb_dim128_l100_units512_emb128_no_pos/category/epoch{epoch}_100000_valid_interval1_gen20'
def load_data(path):
    data = np.loadtxt(path, dtype= int)
    # data = Rm_peaks_steps(data, 100)
    return data
pred_traj = [load_data(pred_path + f'/no_gen_pos_prediction_{i}') for i in range(20)]
for lagtime in range(int(50/interval), int(2001/interval), int(50/interval)):
    msm = MarkovStateModel(lag_time=lagtime, n_timescales=4, verbose=False, reversible_type="transpose")
    msm.fit(list(pred_traj))
    print(lagtime*interval, end=" ")
    for i in range(len(msm.timescales_))[:4]:
        print(msm.timescales_[i]*interval, end=" ")
    print("\n", end="")

50 17610.61845613542 4279.422896608813 1637.0264417309945 715.6383001974009 
100 29413.328762115947 6588.516798034854 2488.556029738148 1108.6647264662192 
150 38290.95870726109 8061.242791509863 3009.5265477419316 1386.0361275725108 
200 44747.28989061273 9095.761912864307 3385.0525495201973 1595.0739540508969 
250 49940.42283297502 9819.0442389384 3658.1145753307446 1752.7554142703823 
300 54051.3065841793 10431.065229719465 3841.221878439059 1859.075438666675 
350 57679.608236296546 10935.753079983186 4023.149095691811 1926.5578179077675 
400 60659.161017898936 11306.153407297374 4166.836918554974 1988.6856239910333 
450 63399.55252919434 11615.013181634793 4287.340893145509 2032.9498302153147 
500 65683.0661094091 11895.131064830697 4362.230674587007 2085.4634699581093 
550 67763.26315027739 12133.354864683468 4446.639011752244 2136.918209983259 
600 69440.48887587112 12354.288268902452 4505.451728519365 2178.515105467728 
650 70846.486900435 12540.696275079175 4572.4912920570605 2

In [12]:
interval = 1
epoch = 40
task = 'Fip35_micro100' 
# task = 'Fip35_macro5' 
# task = 'Fip35_macro'
# task = 'Fip35_micro' 
# pred_path = f'/home/zengwenqi/projects/MD_code2/results/{task}/trans_gpt/Label0.0_window50_interval{interval}_lr0.0005_emb_dim128_l100_block2_scheduled/category/epoch{epoch}_100000_valid_interval{interval}_gen20'
# pred_path = "/home/zengwenqi/projects/MD_code2/results/Fip35_macro/trans_gpt/Label0.0_window50_interval1_lr0.0005_emb_dim128_l100_block2_scheduled_false20/category/epoch40_100000_valid_interval1"
pred_path_lstm = f'/home/zengwenqi/projects/MD_code2/results/Fip35_micro100/lstm/Label0.0_sparse50_interval{interval}_lr0.001_emb_dim128_l100_units512_emb128_no_pos/category/epoch{epoch}_100000_valid_interval1_gen20'
def load_data(path):
    data = np.loadtxt(path, dtype= int)
    # data = Rm_peaks_steps(data, 100)
    return data
pred_traj_lstm = [load_data(pred_path_lstm + f'/no_gen_pos_prediction_{i}') for i in range(20)]
for lagtime in range(int(50/interval), int(2001/interval), int(50/interval)):
    msm = MarkovStateModel(lag_time=lagtime, n_timescales=4, verbose=False, reversible_type="transpose")
    msm.fit(list(pred_traj_lstm))
    print(lagtime*interval, end=" ")
    for i in range(len(msm.timescales_))[:4]:
        print(msm.timescales_[i]*interval, end=" ")
    print("\n", end="")


50 12105.175817355188 1095.725284077208 477.4425692230811 393.00943266196225 
100 13487.51950106684 1682.1003017380415 801.3716198055499 646.0052439106969 
150 14122.08121971667 2122.1408610311355 1059.6032362406538 845.8373631576746 
200 14449.194112955278 2466.465666387103 1281.6921834942916 995.8177685495707 
250 14672.235888163283 2759.9742024283955 1480.63866973403 1122.7987954396106 
300 14855.533228226512 3008.737984245152 1648.3241642365329 1240.3007894732332 
350 15005.053733382809 3208.9457840745113 1798.8452431392366 1325.7416688384974 
400 15130.108476757927 3366.8436734833886 1926.0802188897496 1402.7895399722975 
450 15221.784711946475 3519.4754252415546 2031.7369152300093 1477.0700057261497 
500 15277.675547116205 3625.4230529580723 2130.2545588859707 1543.6257936932705 
550 15343.159407293291 3749.065755433227 2218.3031956746963 1594.0462886205084 
600 15463.942281301972 3837.111356613327 2301.512164099484 1634.8293330089691 
650 15582.191100576845 3892.732308130904 236

## MFPT --micro100

In [10]:
from pyemma import msm
interval = 5
epoch = 40
task = 'Fip35_micro100' 
# task = 'Fip35_macro5' 
# task = 'Fip35_macro'
# task = 'Fip35_micro' 
pred_path= f'/home/zengwenqi/projects/MD_code2/results/{task}/trans_gpt/Label0.0_window50_interval{interval}_lr0.0005_emb_dim128_l100_block2_scheduled/category/epoch{epoch}_100000_valid_interval{interval}_gen20'
def load_data(path):
    data = np.loadtxt(path, dtype= int)
    return data
pred_traj = [load_data(pred_path + f'/no_gen_pos_prediction_{i}') for i in range(20)]

datapath = f'/home/zengwenqi/projects/MD_code2/data/{task}/'
train=np.loadtxt(datapath + 'train', dtype=int).reshape(-1)

msm_nrev = msm.estimate_markov_model(train,500)
msm_nrev_pred = msm.estimate_markov_model(pred_traj, int(500/interval))

0      beta2 folded, beta1 unfolded

1      misfolded

2        beta1 folded, beta2 unfolded

3       unfolded

4     folded / native

see /home/zengwenqi/projects/MD_code2/data/Fip35/unzip_all/macro5-rmsds-report.txt


In [15]:
msm_nrev = msm.estimate_markov_model(train[0:int(len(train)/2)],500)
msm_nrev_pred = msm.estimate_markov_model(pred_traj, int(500/interval))
# macro 0 (3) <---> macro 4 (0)
print(f'macro 0 to macro 4: pred: {msm_nrev_pred.mfpt(3,0)*interval}')
print(f'reverse: pred: {msm_nrev_pred.mfpt(0,3)*interval}')

macro 0 to macro 4: pred: 74863.1711877419
reverse: pred: 2158930.696073911


In [16]:
print(f'macro 0 to macro 4: md: {msm_nrev.mfpt(3,0)}')
print(f'reverse: md: {msm_nrev.mfpt(0,3)}')

macro 0 to macro 4: md: 126251.79449176475
reverse: md: 114922533.92518084


## MFPT -- macro5

In [15]:
from pyemma import msm
interval = 5
epoch = 40

task = 'Fip35_macro5' 
# task = 'Fip35_macro'
# task = 'Fip35_micro' 
pred_path = f'/home/zengwenqi/projects/MD_code2/results/{task}/trans_gpt/Label0.0_window50_interval{interval}_lr0.0005_emb_dim128_l100_block2_scheduled/category/epoch{epoch}_100000_valid_interval{interval}'
def load_data(path):
    data = np.loadtxt(path, dtype= int)
    return data
pred_traj = [load_data(pred_path + f'/no_gen_pos_prediction_{i}') for i in range(11)]

datapath = f'/home/zengwenqi/projects/MD_code2/data/{task}/'
train=np.loadtxt(datapath + 'train', dtype=int).reshape(-1)


msm_nrev = msm.estimate_markov_model(train, 100)
msm_nrev_pred = msm.estimate_markov_model(pred_traj, int(100/interval))

In [17]:
for start in range(1,5):
    for end in range(start):
        if start == end:
            continue
        # end = 4
        print(f'macro {start} to macro {end}macro {start} to macro {end}: pred: {msm_nrev_pred.mfpt(start, end)*interval}')
        print(f'reverse: pred: {msm_nrev_pred.mfpt(end, start)*interval}')

        print(f'macro {start} to macro {end}: md: {msm_nrev.mfpt(start, end)}')
        print(f'reverse: md: {msm_nrev.mfpt(end, start)}')

macro 1 to macro 0: pred: 13436.874054276977
reverse: pred: 205.1380941726018
macro 1 to macro 0: md: 16194.298592287541
reverse: md: 643.3988923130912
macro 2 to macro 0: pred: 23334.215285785038
reverse: pred: 756561.8372379722
macro 2 to macro 0: md: 19876.593323335346
reverse: md: 758762.9220179513
macro 2 to macro 1: pred: 10090.842837575403
reverse: pred: 756550.2007498671
macro 2 to macro 1: md: 3682.2947310478285
reverse: md: 758119.5231256379
macro 3 to macro 0: pred: 26876.15909263992
reverse: pred: 29891.620100998447
macro 3 to macro 0: md: 62189.81052155054
reverse: md: 26836.056370920585
macro 3 to macro 1: pred: 13459.64810006254
reverse: pred: 29706.845068525326
macro 3 to macro 1: md: 46307.099298539375
reverse: md: 26504.244847884176
macro 3 to macro 2: pred: 770009.5600153216
reverse: pred: 39797.3990714994
macro 3 to macro 2: md: 804426.6224241926
reverse: md: 30186.539578931955
macro 4 to macro 0: pred: 112958.63071675361
reverse: pred: 71494.2813486579
macro 4 to m

# MFPT: 100 states

In [126]:
file_path2 = '/home/zengwenqi/projects/MD_code2/data/Fip35/zip_files/100state_5state.log'
df = pd.read_csv(file_path2, delim_whitespace=True)

macro_to_numeric = {
    "macro1": 0,
    "macro2": 1,
    "macro3": 2,
    "macro4": 3,
    "macro5": 4,
}
df_melted = df.melt(id_vars=["#id"], var_name="macro", value_name="value")
filtered = df_melted[df_melted["value"] > 0.5]
filtered["macro"] = filtered["macro"].map(macro_to_numeric)
id_to_macro_mapping2 = filtered[["#id", "macro"]].set_index("#id")["macro"].to_dict()

/tmp/ipykernel_1762457/974508201.py:2: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(file_path2, delim_whitespace=True)
/tmp/ipykernel_1762457/974508201.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered["macro"] = filtered["macro"].map(macro_to_numeric)


In [113]:
import pandas as pd

file_path = "/home/zengwenqi/projects/MD_code2/data/Fip35/mapping-100-to-5.txt"
id_map = np.loadtxt(file_path, dtype=int)

# df = pd.read_csv(file_path, delim_whitespace=True)

# macro_to_numeric = {
#     "macro1": 0,
#     "macro2": 1,
#     "macro3": 2,
#     "macro4": 3,
#     "macro5": 4,
# }
# df_melted = df.melt(id_vars=["#id"], var_name="macro", value_name="value")
# filtered = df_melted[df_melted["value"] > 0.5]
# filtered["macro"] = filtered["macro"].map(macro_to_numeric)
# id_to_macro_mapping = filtered[["#id", "macro"]].set_index("#id")["macro"].to_dict()

# output_path = "/home/zengwenqi/projects/MD_code2/data/Fip35/zip_files/mapping.json"
# with open(output_path, "w") as f:
#     import json
#     json.dump(id_to_macro_mapping, f, indent=4)


In [116]:
id_to_macro_mapping = {i: id_map[i] for i in range(len(id_map))}

In [134]:
for i in range(20):
    print(f'{i} {id_to_macro_mapping2[i]}')

0 4
1 1
2 3
3 0
4 2
5 3
6 3
7 3
8 3
9 1
10 3
11 3
12 4
13 3
14 3
15 3
16 1
17 3
18 1
19 3


In [117]:
id_to_macro_mapping

{0: 4,
 1: 1,
 2: 3,
 3: 0,
 4: 2,
 5: 3,
 6: 3,
 7: 3,
 8: 3,
 9: 1,
 10: 3,
 11: 3,
 12: 4,
 13: 3,
 14: 3,
 15: 3,
 16: 1,
 17: 3,
 18: 1,
 19: 3,
 20: 2,
 21: 4,
 22: 3,
 23: 3,
 24: 1,
 25: 3,
 26: 3,
 27: 3,
 28: 3,
 29: 3,
 30: 1,
 31: 3,
 32: 3,
 33: 3,
 34: 3,
 35: 3,
 36: 3,
 37: 4,
 38: 3,
 39: 3,
 40: 2,
 41: 4,
 42: 3,
 43: 3,
 44: 3,
 45: 3,
 46: 1,
 47: 3,
 48: 3,
 49: 0,
 50: 3,
 51: 3,
 52: 3,
 53: 3,
 54: 3,
 55: 1,
 56: 3,
 57: 3,
 58: 3,
 59: 3,
 60: 3,
 61: 4,
 62: 1,
 63: 3,
 64: 3,
 65: 4,
 66: 3,
 67: 3,
 68: 3,
 69: 3,
 70: 3,
 71: 3,
 72: 3,
 73: 3,
 74: 3,
 75: 2,
 76: 3,
 77: 3,
 78: 3,
 79: 1,
 80: 3,
 81: 3,
 82: 3,
 83: 3,
 84: 0,
 85: 4,
 86: 1,
 87: 3,
 88: 4,
 89: 3,
 90: 3,
 91: 3,
 92: 2,
 93: 3,
 94: 3,
 95: 3,
 96: 4,
 97: 3,
 98: 3,
 99: 3}

In [118]:
# from pyemma import msm
import msmbuilder.msm
import msmbuilder.tpt
interval = 5
epoch = 40

# task = 'Fip35_macro5' 
# task = 'Fip35_macro'
task = 'Fip35_micro100' 
pred_path = f'/home/zengwenqi/projects/MD_code2/results/{task}/trans_gpt/Label0.0_window50_interval{interval}_lr0.0005_emb_dim128_l100_block2_scheduled/category/epoch{epoch}_100000_valid_interval{interval}'
def load_data(path):
    data = np.loadtxt(path, dtype= int)
    return data
pred_traj = [load_data(pred_path + f'/no_gen_pos_prediction_{i}') for i in range(11)]

datapath = f'/home/zengwenqi/projects/MD_code2/data/{task}/'
train=np.loadtxt(datapath + 'train', dtype=int).reshape(-1)


In [ ]:
pred_traj_mapped = [np.array([id_to_macro_mapping[i] for i in traj]) for traj in pred_traj]
pred_traj_lstm_mapped = [np.array([id_to_macro_mapping[i] for i in traj]) for traj in pred_traj_lstm]
train_mapped = np.array([id_to_macro_mapping[i] for i in train])

## GPT

In [56]:
def get_tpm(traj, lagtime):
    _trajs = traj
    _lagtime = lagtime
    _dim = numpy.max(_trajs).astype(int)+1
    _TCM = numpy.zeros((_dim, _dim), dtype=int)
    for i in range(len(_trajs)):
        for t in range(len(_trajs[i])-_lagtime):
            _TCM[int(_trajs[i][t]), int(_trajs[i][t+_lagtime])] += 1


    TCM = (_TCM + _TCM.T) / 2
    TPM = TCM
    for i in range(len(TCM)):
        TPM[i] /= np.sum(TPM[i])
    return TPM

In [119]:
msm_gpt = msmbuilder.msm.MarkovStateModel(n_timescales=10, lag_time=500/interval, 
                                      reversible_type="transpose", verbose=False, ergodic_cutoff="off")
msm_gpt.fit(pred_traj)

MarkovStateModel(ergodic_cutoff='off', lag_time=100.0, n_timescales=10,
                 reversible_type='transpose', verbose=False)

In [91]:
source = 4 # fold
target = 3 # unfold

target_states = [key for key, val in id_to_macro_mapping.items() if val == target]
filter_target_states = [i if i < 5 else i - 1 for i in target_states]
state_mfpt = msmbuilder.tpt.mfpts(msm=msm_gpt, sinks=filter_target_states, lag_time=100)

source_states = [key for key, val in id_to_macro_mapping.items() if val == source]
filter_source_states = [i if i < 5 else i - 1 for i in source_states]
# state_mfpt[filter_source_states].mean()*interval

import numpy as np
from collections import Counter
state_counts = Counter(np.concatenate(pred_traj))
source_weights = np.array([state_counts[state] for state in source_states])
normalized_weights = source_weights / np.sum(source_weights)

source_mfpt_values = state_mfpt[filter_source_states]
weighted_mean_mfpt = np.average(source_mfpt_values, weights=normalized_weights)
weighted_mean_adjusted = weighted_mean_mfpt * interval
print("Weighted mean MFPT (adjusted):", weighted_mean_adjusted)


Weighted mean MFPT (adjusted): 390965.9172972153


In [92]:
source = 3
target = 4

target_states = [key for key, val in id_to_macro_mapping.items() if val == target]
filter_target_states = [i if i < 5 else i - 1 for i in target_states]
state_mfpt = msmbuilder.tpt.mfpts(msm=msm_gpt, sinks=filter_target_states, lag_time=100)

source_states = [key for key, val in id_to_macro_mapping.items() if val == source]
filter_source_states = [i if i < 5 else i - 1 for i in source_states]
# state_mfpt[filter_source_states].mean()*interval

import numpy as np
from collections import Counter
state_counts = Counter(np.concatenate(pred_traj))
source_weights = np.array([state_counts[state] for state in source_states])
normalized_weights = source_weights / np.sum(source_weights)

source_mfpt_values = state_mfpt[filter_source_states]
weighted_mean_mfpt = np.average(source_mfpt_values, weights=normalized_weights)
weighted_mean_adjusted = weighted_mean_mfpt * interval
print("Weighted mean MFPT (adjusted):", weighted_mean_adjusted)

Weighted mean MFPT (adjusted): 60308.43885398753


## MD baseline

In [120]:
msm = msmbuilder.msm.MarkovStateModel(n_timescales=10, lag_time=500, 
                                      reversible_type="transpose", verbose=False, ergodic_cutoff="off")
msm.fit(train)

MarkovStateModel(ergodic_cutoff='off', lag_time=500, n_timescales=10,
                 reversible_type='transpose', verbose=False)

In [121]:
msm.timescales_

array([61416.11600081,  9326.98657749,  1075.69287163,   943.62721729,
         522.80648103,   514.27479351,   375.25761843,   366.02632677,
         356.38951774,   302.17021962])

In [123]:
source = 4 # fold
target = 3 # unfold

target_states = [key for key, val in id_to_macro_mapping.items() if val == target]
this_state_mfpt = msmbuilder.tpt.mfpts(msm=msm, sinks=target_states, lag_time=500)
source_states = [key for key, val in id_to_macro_mapping.items() if val == source]
this_state_mfpt[source_states].mean()


214867.92595216134

In [124]:
source = 3
target = 4

target_states = [key for key, val in id_to_macro_mapping.items() if val == target]
this_state_mfpt = msmbuilder.tpt.mfpts(msm=msm, sinks=target_states, lag_time=500)
source_states = [key for key, val in id_to_macro_mapping.items() if val == source]
this_state_mfpt[source_states].mean()


72587.13892704746

已验证： folding time: 14 microsecond （和science的文章的 unfold -> fold waiting time; folding time 相比）


达到 markovian：100 ns, 10000 states. (不用提)

In [30]:
from pyemma import msm
msm_nrev = msm.estimate_markov_model(train_mapped, 100)
msm_nrev_pred = msm.estimate_markov_model(pred_traj_mapped, int(100/interval))

In [31]:
for start in range(1,5):
    for end in range(start):
        if start == end:
            continue
        # end = 4
        print(f'macro {start} to macro {end}macro {start} to macro {end}: pred: {msm_nrev_pred.mfpt(start, end)*interval}')
        print(f'reverse: pred: {msm_nrev_pred.mfpt(end, start)*interval}')

        print(f'macro {start} to macro {end}: md: {msm_nrev.mfpt(start, end)}')
        print(f'reverse: md: {msm_nrev.mfpt(end, start)}')

macro 1 to macro 0macro 1 to macro 0: pred: 82900.28940872374
reverse: pred: 70779.091078608
macro 1 to macro 0: md: 367716.19061115064
reverse: md: 783434.7832705994
macro 2 to macro 0macro 2 to macro 0: pred: 97028.81183371093
reverse: pred: 147657.73282346484
macro 2 to macro 0: md: 397407.4764975926
reverse: md: 35877.53094370662
macro 2 to macro 1macro 2 to macro 1: pred: 86382.61947902488
reverse: pred: 149132.7387988954
macro 2 to macro 1: md: 816721.9281503615
reverse: md: 39473.38993703054
macro 3 to macro 0macro 3 to macro 0: pred: 73850.69477020798
reverse: pred: 4881.303871548809
macro 3 to macro 0: md: 363908.20310178585
reverse: md: 212.1285160416889
macro 3 to macro 1macro 3 to macro 1: pred: 69361.16061980871
reverse: pred: 12512.96805126417
macro 3 to macro 1: md: 783222.6547545576
reverse: md: 3807.987509365709
macro 3 to macro 2macro 3 to macro 2: pred: 145375.33341543964
reverse: pred: 25777.02152702593
macro 3 to macro 2: md: 35665.40242766492
reverse: md: 33499.27

In [11]:
interval

1

In [12]:
from pyemma import msm
msm_nrev = msm.estimate_markov_model(train_mapped, 100)
msm_nrev_pred = msm.estimate_markov_model(pred_traj_lstm_mapped, int(100/interval))

import pandas as pd

# Assuming the following objects are already available:
# msm_nrev_pred: The predicted MSM (non-reversible version)
# msm_nrev: The MSM trained on the original MD trajectories
# interval: The time interval for scaling

# Initialize an empty list to store the results
results = []

# Loop over start and end states
for start in range(1, 5):
    for end in range(start):
        if start == end:
            continue
        # Calculate MFPTs
        pred_forward = msm_nrev_pred.mfpt(start, end) * interval
        pred_reverse = msm_nrev_pred.mfpt(end, start) * interval
        md_forward = msm_nrev.mfpt(start, end)
        md_reverse = msm_nrev.mfpt(end, start)

        # Add forward transition results
        results.append({
            "Transition": f"macro{start}_to_macro{end}",
            "MD_MFPT": md_forward,
            "Pred_MFPT": pred_forward
        })

        # Add reverse transition results
        results.append({
            "Transition": f"macro{end}_to_macro{start}",
            "MD_MFPT": md_reverse,
            "Pred_MFPT": pred_reverse
        })

# Convert the results to a pandas DataFrame
results_df = pd.DataFrame(results)

# Display the DataFrame
print(results_df)

# Optionally save to a CSV file
results_df.to_csv("mfpt_results_formatted.csv", index=False)

          Transition        MD_MFPT     Pred_MFPT
0   macro1_to_macro0  367716.190611  1.229446e+06
1   macro0_to_macro1  783434.783271  2.036089e+05
2   macro2_to_macro0  397407.476498  1.223438e+06
3   macro0_to_macro2   35877.530944  2.922640e+04
4   macro2_to_macro1  816721.928150  2.099093e+05
5   macro1_to_macro2   39473.389937  4.153565e+04
6   macro3_to_macro0  363908.203102  1.230746e+06
7   macro0_to_macro3     212.128516  9.494637e+03
8   macro3_to_macro1  783222.654755  1.966018e+05
9   macro1_to_macro3    3807.987509  1.187875e+03
10  macro3_to_macro2   35665.402428  4.331546e+04
11  macro2_to_macro3   33499.273396  1.627518e+04
12  macro4_to_macro0  464738.600576  1.244791e+06
13  macro0_to_macro4   30154.571375  1.146612e+04
14  macro4_to_macro1  884053.052229  2.237554e+05
15  macro1_to_macro4   33750.430369  1.626799e+04
16  macro4_to_macro2   93646.641140  4.673753e+04
17  macro2_to_macro4   20592.557493  7.623724e+03
18  macro4_to_macro3  100830.397474  2.874857e+04
